In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import MinMaxScaler

operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/GE/step-1'
output_step2_path='../../Data/output/GE/step-2'
output_step3_path='../../Data/output/GE/step-3'

#Read the files
index_walkability = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index_walkability = index_walkability.to_crs(operation_crs)

zones_girec = gpd.read_file(f'{input_file_path}/network_agreg/GEO_GIREC-SHP/GEO_GIREC.shp')
zones_girec = zones_girec.to_crs(operation_crs)

agglo_carreau = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_CARREAU_200-SHP/AGGLO_CARREAU_200.shp')
agglo_carreau = agglo_carreau.to_crs(operation_crs)

zones_communes = gpd.read_file(f'{input_file_path}/network_agreg/CAD_COMMUNE-SHP/CAD_COMMUNE.shp')
zones_communes = zones_communes.to_crs(operation_crs)

zones_communes_GE_fusionnee = gpd.read_file(f'{input_file_path}/network_agreg/CAD_COMMUNE-SHP/CAD_COMMUNES_GE_fusionnee.shp')
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.to_crs(operation_crs)

canton_GE = gpd.read_file(f'{input_file_path}/network_agreg/CANTON_GE/CANTON_POLYGON.shp')
canton_GE = canton_GE.to_crs(operation_crs)

lac = gpd.read_file(f'{input_file_path}/network_agreg/LAC/LAC_LEMAN_WITHOUT_BRIDGE-SHP/LAC_LEMAN_WITHOUT_BRIDGE.shp')
lac = lac.to_crs(operation_crs)


# [GIREC](https://sitg.ge.ch/donnees/geo-girec)

In [ ]:
# Spatial join
segments_girec = gpd.sjoin(index_walkability, zones_girec, how="inner", predicate="within")

# Columns to aggregate
cols = index_walkability.columns.to_list()

def weighted_mean(df, cols, weight_col):
    return (df[cols].multiply(df[weight_col], axis=0).sum() / df[weight_col].sum())

cols_to_agg = cols[3:]  # tes colonnes d'indicateurs

# Calcul pondéré
girec_stats = (
    segments_girec
        .groupby("OBJECTID")
        .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
        .reset_index()
)

# Merge back with zones_mmt polygons
zones_girec = zones_girec.merge(girec_stats, on="OBJECTID", how="left")
zones_girec

#Drop nan values 
zones_girec = zones_girec.dropna(subset=["walk_index"])

#normalisation
scaler = MinMaxScaler()
zones_girec["walk_index_norm"] = scaler.fit_transform(zones_girec[["walk_index"]])

In [ ]:
print(f"walk_index NaN      : {zones_girec['walk_index'].isna().sum()}")
print(f"walk_index_norm NaN : {zones_girec['walk_index_norm'].isna().sum()}")
print(f"Index continu ?     : {zones_girec.index.is_monotonic_increasing}")
print(f"Index reset ?       : {list(zones_girec.index[:5])}")

In [ ]:
zones_girec

In [ ]:
zones_girec = zones_girec.merge(
    zones_communes[["NO_COM_FED", "COMMUNE"]].drop_duplicates(subset="NO_COM_FED"),
    on="NO_COM_FED",
    how="left"
)

# déplacer COMMUNE en 3ème position
cols = zones_girec.columns.tolist()
cols.insert(2, cols.pop(cols.index('COMMUNE')))
zones_girec = zones_girec[cols]

In [ ]:
zones_girec

In [ ]:
zones_girec.dtypes

In [ ]:
zones_girec["walk_index"].isna().sum()

# Carreau 200

In [ ]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_carreau = gpd.sjoin(index_walkability, agglo_carreau, how="inner", predicate="within")

# Aggregate by mean
carreau_stats = (
    segments_carreau
    .groupby("GRID_ID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
agglo_carreau = agglo_carreau.merge(carreau_stats, on="GRID_ID", how="left")

# Drop rows with missing values (optional)
agglo_carreau = agglo_carreau.dropna(subset=["walk_index"])

#normalisation
scaler = MinMaxScaler()
agglo_carreau["walk_index_norm"] = scaler.fit_transform(agglo_carreau[["walk_index"]])

In [ ]:
agglo_carreau

In [ ]:
agglo_carreau['GRID_ID'].unique()

# [Communes](https://sitg.ge.ch/donnees/cad-commune)

In [ ]:
zones_communes.head()

In [ ]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_communes = gpd.sjoin(index_walkability, zones_communes, how="inner", predicate="within")

# Aggregate by mean
communes_stats = (
    segments_communes
    .groupby("OBJECTID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back with commune area
zones_communes = zones_communes.merge(communes_stats, on="OBJECTID", how="left")

# Drop rows with missing values (optional)
zones_communes = zones_communes.dropna(subset=["walk_index"])

#normalisation
scaler = MinMaxScaler()
zones_communes["walk_index_norm"] = scaler.fit_transform(zones_communes[["walk_index"]])

In [ ]:
zones_communes.head()

*Communes avec Genève fusionnée (1 seul polygone au lieu de 4 différents)*

In [ ]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_communes_GE_fusionnee = gpd.sjoin(index_walkability, zones_communes_GE_fusionnee, how="inner", predicate="within")

# Aggregate by mean
communes_GE_fusionnee_stats = (
    segments_communes_GE_fusionnee
    .groupby("OBJECTID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.merge(communes_GE_fusionnee_stats, on="OBJECTID", how="left")

# Drop rows with missing values (optional)
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.dropna(subset=["walk_index"])

#normalisation
scaler = MinMaxScaler()
zones_communes_GE_fusionnee["walk_index_norm"] = scaler.fit_transform(zones_communes_GE_fusionnee[["walk_index"]])

In [ ]:
zones_communes_GE_fusionnee.head()

# STATISTICS

## COMMUNE

### [POPULATION 2025 - COMMUNE](https://statistique.ge.ch/atlas/index.php#c=indicator&i=population.pop_tot&s=2025&t=A01&view=map3)

In [ ]:
pop_communes = pd.read_csv(f'{input_file_path}/STAT/POPULATION/pop_2025.csv', sep=";", header=2)
pop_communes = pop_communes.rename(columns={"Population résidante 2025":"pop_2025"})

colonnes_pop = ['pop_2025']

pop_communes[colonnes_pop] = pop_communes[colonnes_pop].apply(pd.to_numeric, errors='coerce')


In [ ]:
pop_communes.head()

In [ ]:
zones_communes_GE_fusionnee = (zones_communes_GE_fusionnee.merge(
    pop_communes[["Code"] + colonnes_pop],
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
).drop(columns="Code"))

In [ ]:
zones_communes_GE_fusionnee.head()

### [FREQUENCE CRIMINALITE 2024 - COMMUNE](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)

In [ ]:
criminalite_communes = pd.read_csv(f'{input_file_path}/STAT/CRIMINALITE/CRIMINALITE_GE_2024.csv', sep=";", header=2)

criminalite_communes = criminalite_communes.rename(columns={"Loi sur les stupéfiants (LStup) : fréquence d'infractions 2024":"freq_LStup_infra","Code pénal (CP) : fréquence d'infractions 2024":"freq_CP_infra", "Loi sur les étrangers et l’intégration (LEI) : fréquence d'infractions 2024":"freq_LEI_infra"})

#convert string to float
colonnes_crim = ['freq_LStup_infra', 'freq_CP_infra', 'freq_LEI_infra']

criminalite_communes[colonnes_crim] = criminalite_communes[colonnes_crim].apply(pd.to_numeric, errors='coerce')

criminalite_communes['freq_crim_mean'] = criminalite_communes[colonnes_crim].mean(axis=1)

#print(criminalite_communes[colonnes].dtypes)

In [ ]:
criminalite_communes.head()

In [ ]:
zones_communes_GE_fusionnee = (zones_communes_GE_fusionnee.merge(
    criminalite_communes[["Code"]+ colonnes_crim + ["freq_crim_mean"]],
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
).drop(columns="Code"))

In [ ]:
zones_communes_GE_fusionnee.head()

### [TAUX MOTORISATION 2024 - COMMUNE](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)

In [ ]:
taux_motorisation_communes = pd.read_csv(f'{input_file_path}/STAT/TAUX_MOTORISATION/taux_motorisation_2024.csv', sep=";", header=2)

In [ ]:
taux_motorisation_communes.head()

In [ ]:
taux_motorisation_communes = taux_motorisation_communes.rename(columns={"Taux de motorisation 2024": "freq_voitures_24"})

#convert string to float
colonnes_motor = ["freq_voitures_24"]

taux_motorisation_communes[colonnes_motor] = taux_motorisation_communes[colonnes_motor].apply(pd.to_numeric, errors='coerce')

In [ ]:
taux_motorisation_communes.head()

In [ ]:
zones_communes_GE_fusionnee = (zones_communes_GE_fusionnee.merge(
    taux_motorisation_communes[["Code"] + colonnes_motor],
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
).drop(columns="Code"))

In [ ]:
zones_communes_GE_fusionnee.head()

### [CHOMAGE 2025 - COMMUNE](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)

In [ ]:
chomage_communes = pd.read_csv(f'{input_file_path}/STAT/CHOMAGE/chomage_2025.csv', sep=";", header=2)

In [ ]:
chomage_communes.head()

In [ ]:
chomage_communes = chomage_communes.rename(columns={"Chômeurs inscrits 2025": "nb_chomage"})

#convert string to float
colonnes_chomage = ["nb_chomage"]

chomage_communes[colonnes_chomage] = chomage_communes[colonnes_chomage].apply(pd.to_numeric, errors='coerce')

In [ ]:
chomage_communes.head()

In [ ]:
zones_communes_GE_fusionnee = (zones_communes_GE_fusionnee.merge(
    chomage_communes[["Code"] + colonnes_chomage],
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
).drop(columns="Code"))

In [ ]:
zones_communes_GE_fusionnee.head()

In [ ]:
zones_communes_GE_fusionnee["freq_chomage"] = (
    zones_communes_GE_fusionnee["nb_chomage"] /
    zones_communes_GE_fusionnee["pop_2025"]
).replace([float("inf")], pd.NA) * 1000

zones_communes_GE_fusionnee.drop(columns="nb_chomage", inplace=True)

## GIREC

### [PRECARITE 2024 - GIREC](https://ise.unige.ch/cati-ge/#bbox=2496349,1120369,6343,4432&c=indicator&i=cati_spec.res_class&s=2024&view=map1)

In [ ]:
precarite_GIREC = pd.read_csv(f'{input_file_path}/STAT/PRECARITE/indicateurs_precarite_GIREC_2024.csv', sep=";", header=2)

precarite_GIREC = precarite_GIREC.rename(columns={"IB1 quartiles 2024": "IB1_quartiles_24",
                                                  "IB2 quartiles 2024": "IB2_quartiles_24",
                                                  "IB3 quartiles 2024": "IB3_quartiles_24",
                                                  "IB4 quartiles 2024": "IB4_quartiles_24",
                                                  "IB5 quartiles 2024": "IB5_quartiles_24",
                                                  "IB6 quartiles 2024": "IB6_quartiles_24",
                                                  "Nombre de critères 2024": "precarite_score_24"})

ib_cols = ['IB1_quartiles_24', 'IB2_quartiles_24', 'IB3_quartiles_24',
           'IB4_quartiles_24', 'IB5_quartiles_24', 'IB6_quartiles_24']
for col in ib_cols:
    precarite_GIREC[col] = precarite_GIREC[col].astype(str).replace({'-888': 'pas d\'habitants', '-999': 'secret stat.'})

precarite_GIREC["precarite_score_24"] = pd.to_numeric(precarite_GIREC["precarite_score_24"].where(~precarite_GIREC["precarite_score_24"].str.startswith("N/A"), other=None), errors='coerce')

precarite_GIREC["no_commune_fed"] = precarite_GIREC["Code"].str[6:10].astype(int)
precarite_GIREC["code_sect"] = precarite_GIREC["Code"].str[13:15]
precarite_GIREC["cd_ss_secteur"] = precarite_GIREC["Code"].str[16:19]

In [ ]:
precarite_GIREC.head()

In [ ]:
precarite_GIREC.dtypes

In [ ]:
zones_girec = zones_girec.merge(
    precarite_GIREC[["Code","no_commune_fed", "code_sect", "cd_ss_secteur", "precarite_score_24"]+ib_cols],
    left_on=["NO_COM_FED", "CODE_SECT", "CD_SS_SECT"],
    right_on=["no_commune_fed", "code_sect", "cd_ss_secteur"],
    how="left"
).drop(columns=["no_commune_fed", "code_sect", "cd_ss_secteur"])
zones_girec = zones_girec.rename(columns={"Code": "CODE"})


In [ ]:
zones_girec

### [TRAVAIL / CHOMAGE - GIREC](https://ise.unige.ch/cati-ge/#bbox=2484722,1128676,29221,18908&c=indicator&i=cati_globale.delta_emptot_tot&i2=cati_globale.pct_chom&s=2024&s2=2024&t=A02&t2=A02&view=map1)

In [ ]:
travail_chomage_GIREC = pd.read_csv(f'{input_file_path}/STAT/CHOMAGE/emploi_chomage_2024_girec.csv', sep=";", header=2)
travail_chomage_GIREC = travail_chomage_GIREC.rename(columns={"Delta emplois totaux 2024": "delta_emploi_24", "Part de chômeurs inscrits 2024": "freq_chomeurs_inscrits_24", "Chômeurs inscrits 2024": "chomeurs_inscrits_24", "Emplois totaux 2024": "emplois_totaux_24"})

cols_to_clean = ["delta_emploi_24", "freq_chomeurs_inscrits_24", "chomeurs_inscrits_24", "emplois_totaux_24"]
for col in cols_to_clean:
    travail_chomage_GIREC[col] = pd.to_numeric(travail_chomage_GIREC[col].where(~travail_chomage_GIREC[col].astype(str).str.startswith("N/A"), other=None), errors='coerce')

In [ ]:
travail_chomage_GIREC.head()

In [ ]:
travail_chomage_GIREC.dtypes

In [ ]:
zones_girec = zones_girec.merge(
    travail_chomage_GIREC[["Code","delta_emploi_24", "freq_chomeurs_inscrits_24", "chomeurs_inscrits_24", "emplois_totaux_24"]],
    left_on=["CODE"],
    right_on=["Code"],
    how="left"
).drop(columns=["Code"])

In [ ]:
zones_girec

### [EDUCATION - GIREC](https://ise.unige.ch/cati-ge/#bbox=2488838,1125800,22128,14318&c=indicator&i=cati_globale.edu_el_mod&i2=cati_globale.pct_el_mod&s=2024&s2=2024&t=A02&t2=A02&view=map1)

In [ ]:
education_GIREC = pd.read_csv(f'{input_file_path}/STAT/EDUCATION/education_2024.csv', sep=";", header=2)
education_GIREC = education_GIREC.rename(columns={"Elèves origine modeste 2024": "eleve_origine_modeste_24", "Part d'élèves origine modeste 2024": "freq_eleve_origine_modeste_24"})

cols_to_clean = ["eleve_origine_modeste_24", "freq_eleve_origine_modeste_24"]
for col in cols_to_clean:
    education_GIREC[col] = pd.to_numeric(education_GIREC[col].where(~education_GIREC[col].astype(str).str.startswith("N/A"), other=None), errors='coerce')

In [ ]:
education_GIREC

In [ ]:
zones_girec = zones_girec.merge(
    education_GIREC[["Code","eleve_origine_modeste_24", "freq_eleve_origine_modeste_24"]],
    left_on=["CODE"],
    right_on=["Code"],
    how="left"
).drop(columns=["Code"])

In [ ]:
zones_girec

### [SOCIAL - GIREC](https://ise.unige.ch/cati-ge/#bbox=2488838,1125800,22128,14318&c=indicator&view=map1)

In [ ]:
social_GIREC = pd.read_csv(f'{input_file_path}/STAT/SOCIAL/subsides_sociaux_2024.csv', sep=";", header=2)
social_GIREC = social_GIREC.rename(columns={"Subsides sociaux 2024": "subsides_sociaux_24", "Part bénéf. subs. sociaux 2024": "part_benef_subs_sociaux_24"})

cols_to_clean = ["subsides_sociaux_24", "part_benef_subs_sociaux_24"]
for col in cols_to_clean:
    social_GIREC[col] = pd.to_numeric(social_GIREC[col].where(~social_GIREC[col].astype(str).str.startswith("N/A"), other=None), errors='coerce')

In [ ]:
social_GIREC

In [ ]:
zones_girec = zones_girec.merge(
    social_GIREC[["Code","subsides_sociaux_24", "part_benef_subs_sociaux_24"]],
    left_on=["CODE"],
    right_on=["Code"],
    how="left"
).drop(columns=["Code"])

In [ ]:
zones_girec

### [LOGEMENTS - GIREC](https://ise.unige.ch/cati-ge/#bbox=2488838,1125800,22128,14318&c=indicator&view=map1)

In [ ]:
logement_GIREC = pd.read_csv(f'{input_file_path}/STAT/LOGEMENT/logement_girec_2024.csv', sep=";", header=2)
logement_GIREC = logement_GIREC.rename(columns={"Allocations de logement 2024": "alloc_logement_24", "Part bénéf. alloc. logement 2024": "part_benef_alloc_logement_24"})

cols_to_clean = ["alloc_logement_24", "part_benef_alloc_logement_24"]
for col in cols_to_clean:
    logement_GIREC[col] = pd.to_numeric(logement_GIREC[col].where(~logement_GIREC[col].astype(str).str.startswith("N/A"), other=None), errors='coerce')

In [ ]:
logement_GIREC

In [ ]:
zones_girec = zones_girec.merge(
    logement_GIREC[["Code","alloc_logement_24", "part_benef_alloc_logement_24"]],
    left_on=["CODE"],
    right_on=["Code"],
    how="left"
).drop(columns=["Code"])

In [ ]:
zones_girec

### [REVENU - GIREC](https://ise.unige.ch/cati-ge/#c=indicator&i=cati_globale.rev_contr_bas&i2=cati_globale.pct_rev_bas&s=2024&s2=2024&t=A02&t2=A02&view=map1)

In [ ]:
revenu_GIREC = pd.read_csv(f'{input_file_path}/STAT/REVENU/revenu_girec_2024.csv', sep=";", header=2)
revenu_GIREC = revenu_GIREC.rename(columns={"Bas revenus 2024": "bas_revenus_24", "Part des bas revenus 2024": "part_bas_revenus_24", "Revenu médian 2024":"revenus_median_24"})

cols_to_clean = ["bas_revenus_24", "part_bas_revenus_24", "revenus_median_24"]
for col in cols_to_clean:
    revenu_GIREC[col] = pd.to_numeric(revenu_GIREC[col].where(~revenu_GIREC[col].astype(str).str.startswith("N/A"), other=None), errors='coerce')

In [ ]:
revenu_GIREC

In [ ]:
zones_girec = zones_girec.merge(
    revenu_GIREC[["Code","bas_revenus_24", "part_bas_revenus_24", "revenus_median_24"]],
    left_on=["CODE"],
    right_on=["Code"],
    how="left"
).drop(columns=["Code"])

In [ ]:
zones_girec

### [POPULATION - GIREC](https://statistique.ge.ch/atlas/index.php#c=indicator&i=population.pop_km2&s=2025&t=A01&view=map1)

In [ ]:
pop_GIREC = pd.read_csv(f'{input_file_path}/STAT/POPULATION/pop_GIREC_2025.csv', sep=";", header=2)
pop_GIREC = pop_GIREC.rename(columns={"Densité de la population 2025":"densite_pop_25", 
                                            "Population résidante 2025":"pop_residante_25", 
                                            "0 -19 ans 2025": "0_19_ans_25",
                                            "80 ans ou plus 2025":"80_ou_plus_25", 
                                            "20 - 64 ans 2025":"20_64_ans_25", 
                                            "65 ans ou plus 2025":"65_ou_plus_25", 
                                            "Taux de croissance annuel 2024-2025":"taux_croissance_pop_24_25"})

cols_to_clean = ["densite_pop_25",
                 "pop_residante_25", 
                 "0_19_ans_25", 
                 "80_ou_plus_25",
                 "20_64_ans_25", 
                 "65_ou_plus_25", 
                 "taux_croissance_pop_24_25"]

for col in cols_to_clean:
    pop_GIREC[col] = pd.to_numeric(pop_GIREC[col].where(~pop_GIREC[col].astype(str).str.startswith("N/A"), other=None), errors='coerce')

In [ ]:
pop_GIREC

In [ ]:
pop_GIREC.dtypes

In [ ]:
pop_GIREC["Code"] = pop_GIREC["Code"].astype(str)

zones_girec = zones_girec.merge(
    pop_GIREC[["Code"]+ cols_to_clean],
    left_on=["NUMERO"],
    right_on=["Code"],
    how="left"
).drop(columns=["Code"])

In [ ]:
zones_girec

### [HABITATIONS - GIREC](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map1)

In [ ]:
habitations_GIREC = pd.read_csv(f'{input_file_path}/STAT/TERRITOIRE/habitations_GIREC_2025.csv', sep=";", header=2)
habitations_GIREC = habitations_GIREC.rename(columns={"Parc de bâtiments à usage d'habitation 2025":"tot_habitations_25", 
                                            "Répartition des bâtiments  à usage d'habitation selon le type 2025 - Habitation de 20 logements ou plus":"hab_20_plus_25", 
                                            "Répartition des bâtiments  à usage d'habitation selon le type 2025 - Habitations de moins de 20 logements": "hab_20_moins_25",
                                            "Répartition des bâtiments  à usage d'habitation selon le type 2025 - Maisons individuelles":"maison_indiv_25"})

cols_to_clean = ["tot_habitations_25",
                "hab_20_plus_25",
                "hab_20_moins_25",
                "maison_indiv_25"]

for col in cols_to_clean:
    habitations_GIREC[col] = pd.to_numeric(habitations_GIREC[col].where(~habitations_GIREC[col].astype(str).str.startswith("N/A"), other=None), errors='coerce')

In [ ]:
habitations_GIREC

In [ ]:
habitations_GIREC["Code"] = habitations_GIREC["Code"].astype(str)

zones_girec = zones_girec.merge(
    habitations_GIREC[["Code"]+ cols_to_clean],
    left_on=["NUMERO"],
    right_on=["Code"],
    how="left"
).drop(columns=["Code"])

In [ ]:
zones_girec

### [TAUX MOTORISATION 2024 - GIREC (cf. voir COMMUNE)](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)

In [ ]:
zones_girec = (zones_girec.merge(
    taux_motorisation_communes[["Code"] + colonnes_motor],
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
).drop(columns="Code"))

In [ ]:
zones_girec['freq_voitures_24'].isna().sum()

# CORRELATION

## ANALYSES STAT COMMUNES

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

colonnes_stat = ['walk_index_norm', 
                 'freq_LStup_infra', 
                 'freq_CP_infra', 
                 'freq_LEI_infra', 
                 #'freq_crim_mean', 
                 'freq_voitures_24',
                 'freq_chomage']

zones_communes_GE_fusionnee_corr = zones_communes_GE_fusionnee[colonnes_stat].corr(method='pearson')
#print(zones_communes_corr)

sns.heatmap(zones_communes_GE_fusionnee_corr, annot=True, cmap='coolwarm')
plt.title("Pearson Correlation Heatmap - Communes")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for var in colonnes_stat:
    sns.regplot(data=zones_communes_GE_fusionnee, x=var, y='walk_index_norm')
    plt.title(f"Relation entre {var} et walk_index")
    plt.show()

In [ ]:
import statsmodels.api as sm

variables = [ 'freq_LStup_infra', 
                 'freq_CP_infra', 
                 'freq_LEI_infra', 
                 #'freq_crim_mean', 
                 'freq_voitures_24',
                 'freq_chomage']

data_reg = zones_communes_GE_fusionnee[variables + ["walk_index_norm"]].dropna()

communes_Nan = zones_communes_GE_fusionnee[['COMMUNE'] + colonnes_stat].loc[zones_communes_GE_fusionnee[colonnes_stat].isna().any(axis=1)]
communes_Nan.head()

In [ ]:
X = data_reg[variables]
y = data_reg['walk_index_norm']

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

data_rf = zones_communes_GE_fusionnee[variables + ['walk_index_norm']].dropna()

X = data_rf[variables]
y = data_rf['walk_index_norm']

model = RandomForestRegressor(
    n_estimators=1000,
    random_state=42
)

model.fit(X, y)

importance = pd.Series(
    model.feature_importances_,
    index=variables
).sort_values(ascending=False)

print(importance)

In [ ]:
import matplotlib.pyplot as plt

importance.sort_values().plot.barh()

plt.xlabel("Importance")
plt.title("Importance des variables expliquant le walk_index")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt



data_plot = zones_communes_GE_fusionnee[colonnes_stat].dropna()

In [ ]:
sns.pairplot(
    data_plot,
    kind="reg",
    diag_kind="kde"
)

plt.show()

In [ ]:
zones_communes_GE_fusionnee["type_commune"] = "Autres"
zones_communes_GE_fusionnee.loc[
    zones_communes_GE_fusionnee["COMMUNE"] == "Genève",
    "type_commune"
] = "Genève"

In [ ]:
sns.pairplot(
    zones_communes_GE_fusionnee[variables + ["type_commune"]].dropna(),
    hue="type_commune",
    kind="reg",
    diag_kind="kde"
)

plt.show()

## ANALYSES STAT GIREC

### Analyses de corrélation — zones GIREC

Analyse des relations entre `walk_index` et les indicateurs socio-économiques ajoutés.

In [ ]:
# Récupère toutes les colonnes à partir de 'precarite_score' (excl. geometry)
cols_all = [c for c in zones_girec.columns if c != 'geometry']
start_col = 'precarite_score_24'
cols_to_exclude = ['freq_chomeurs_inscrits_24', 'freq_eleve_origine_modeste_24', 'part_benef_subs_sociaux_24','part_benef_alloc_logement_24', 'part_bas_revenus_24', 'tot_habitations_25', 'pop_residante_25'] + ib_cols
stat_vars = [d for d in cols_all[cols_all.index(start_col):] if d not in cols_to_exclude]


print(f'{len(stat_vars)} variables sélectionnées :')
print(stat_vars)

data_girec = zones_girec[['walk_index_norm'] + stat_vars].dropna()
print(f'\n{len(data_girec)} sous-secteurs avec données complètes / {len(zones_girec)} total')

### 1. Heatmap de corrélation

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = data_girec.corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr,
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    linewidths=0.5, ax=ax
)
ax.set_title('Heatmap de corrélation — zones GIREC')
plt.tight_layout()
plt.show()

### 2. VIF — détection de multicolinéarité

Un VIF > 5 signale une colinéarité problématique entre variables.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

X_vif = data_girec[stat_vars].copy()
vif = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)

print(vif.to_string(index=False))

### 3. OLS — régression linéaire

In [ ]:
import statsmodels.api as sm

X_ols = sm.add_constant(data_girec[stat_vars])
y_ols = data_girec['walk_index_norm']

ols_model = sm.OLS(y_ols, X_ols).fit()
print(ols_model.summary())

### 4. LASSO — sélection automatique de variables

Le LASSO pénalise les coefficients et force les variables non-informatives à 0.

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()
X_lasso = scaler.fit_transform(data_girec[stat_vars])
y_lasso = data_girec['walk_index_norm'].values

lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_lasso, y_lasso)

lasso_coefs = pd.Series(lasso.coef_, index=stat_vars).sort_values(key=abs, ascending=False)
print(f'Alpha optimal : {lasso.alpha_:.4f}')
print(lasso_coefs)

lasso_coefs.sort_values().plot.barh(figsize=(8, 6))
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient LASSO (standardisé)')
plt.title('LASSO — contribution des variables à walk_index')
plt.tight_layout()
plt.show()

### 5. Moran's I — autocorrélation spatiale des résidus OLS

Si les résidus sont spatialement autocorrélés, les p-values OLS sont invalides.

In [ ]:
import libpysal
from esda.moran import Moran

# Matrice de contiguïté spatiale (Queen)
zones_girec_clean = zones_girec[['geometry'] + ['walk_index_norm'] + stat_vars].dropna()
w = libpysal.weights.Queen.from_dataframe(zones_girec_clean)
w.transform = 'r'

# Résidus OLS sur le sous-ensemble sans NaN
X_m = sm.add_constant(zones_girec_clean[stat_vars])
y_m = zones_girec_clean['walk_index_norm']
residuals = sm.OLS(y_m, X_m).fit().resid

moran = Moran(residuals.values, w)
print(f"Moran's I : {moran.I:.4f}")
print(f'p-value    : {moran.p_sim:.4f}')
print('→ Autocorrélation spatiale significative' if moran.p_sim < 0.05 else '→ Pas d\'autocorrélation spatiale significative')

### 6. Random Forest — importance des variables

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X_rf = data_girec[stat_vars]
y_rf = data_girec['walk_index_norm']

rf = RandomForestRegressor(n_estimators=1000, random_state=42)
rf.fit(X_rf, y_rf)

importance_girec = pd.Series(rf.feature_importances_, index=stat_vars).sort_values(ascending=False)
print(importance_girec)

importance_girec.sort_values().plot.barh(figsize=(8, 6))
plt.xlabel('Importance')
plt.title('Random Forest — importance des variables (zones GIREC)')
plt.tight_layout()
plt.show()

### 7. Pairplot — top 6 variables corrélées à walk_index

In [ ]:
# Sélection des 6 variables les plus corrélées à walk_index
top_vars = corr['walk_index_norm'].drop('walk_index_norm').abs().sort_values(ascending=False).head(6).index.tolist()

sns.pairplot(
    data_girec[['walk_index_norm'] + top_vars],
    kind='reg',
    diag_kind='kde'
)
plt.suptitle('Pairplot — walk_index vs top 6 variables (zones GIREC)', y=1.02)
plt.show()

## Zones d'action prioritaires

 Croisement marchabilité × besoin social pour identifier les sous-secteurs prioritaires.

### 1. Scatter walk_index × score précarité 

In [ ]:
import plotly.express as px

df_plot = zones_girec[['walk_index_norm', 'precarite_score_24', 'revenus_median_24', 'chomeurs_inscrits_24', 'NOM', 'COMMUNE', 'CODE']].dropna()
#df_plot['NO_COM_FED'] = df_plot['NO_COM_FED'].astype(str)

fig = px.scatter(
    df_plot,
    x='walk_index_norm',
    y='precarite_score_24',
    color='COMMUNE',
    hover_data={
        'NOM': True,
        'CODE': True,
        'COMMUNE': True,
        'walk_index_norm': ':.3f',
        'precarite_score_24': True,
        'chomeurs_inscrits_24': True
    },
    labels={
        'walk_index_norm': 'Walk Index',
        'precarite_score_24': 'Score de precarité',
        'COMMUNE': 'Commune'
    },
    title='Marchabilité vs Precarité par sous-secteur GIREC',
    opacity=0.8,
    width=950,
    height=650
)

# Lignes médianes
fig.add_hline(y=df_plot['precarite_score_24'].median(), line_dash='dash', line_color='gray', line_width=1)
fig.add_vline(x=df_plot['walk_index_norm'].median(), line_dash='dash', line_color='gray', line_width=1)

# Annotations quadrants
med_w = df_plot['walk_index_norm'].median()
med_r = df_plot['precarite_score_24'].median()
fig.add_annotation(x=df_plot['walk_index_norm'].min(), y=df_plot['precarite_score_24'].max(),
    text="Faible marcha. · Precarité élevée", showarrow=False, font=dict(color='steelblue', size=10), xanchor='left')
fig.add_annotation(x=med_w * 1.01, y=df_plot['precarite_score_24'].max(),
    text="Bonne marcha. · Precarité élevée", showarrow=False, font=dict(color='green', size=10), xanchor='left')
fig.add_annotation(x=df_plot['walk_index_norm'].min(), y=df_plot['precarite_score_24'].min(),
    text="⚑ Faible marcha. · Precarité faible", showarrow=False, font=dict(color='red', size=10), xanchor='left')
fig.add_annotation(x=med_w * 1.01, y=df_plot['precarite_score_24'].min(),
    text="Bonne marcha. · Precarité faible", showarrow=False, font=dict(color='orange', size=10), xanchor='left')

fig.show()

In [ ]:
import matplotlib.pyplot as plt

df_plot = zones_girec[['walk_index_norm', 'precarite_score_24', 'NOM', 'COMMUNE']].dropna()

fig, ax = plt.subplots(figsize=(10, 7))

# Points colorés par commune
communes = df_plot['COMMUNE'].unique()
cmap = plt.cm.get_cmap('tab20', len(communes))
color_map = {c: cmap(i) for i, c in enumerate(communes)}

for commune, grp in df_plot.groupby('COMMUNE'):
    ax.scatter(grp['walk_index_norm'], grp['precarite_score_24'],
               color=color_map[commune], label=commune, alpha=0.7, s=40)

# Lignes médianes
ax.axvline(df_plot['walk_index_norm'].median(), color='gray', linestyle='--', linewidth=0.8)
ax.axhline(df_plot['precarite_score_24'].median(), color='gray', linestyle='--', linewidth=0.8)

# Annotations quadrants
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
mx = df_plot['walk_index_norm'].median()
my = df_plot['precarite_score_24'].median()
ax.text(xmin, ymax * 0.97, 'Faible marcha.\nFort besoin\n→ PRIORITAIRE',
        fontsize=8, color='red', fontweight='bold', va='top')
ax.text(mx * 1.01, ymax * 0.97, 'Bonne marcha.\nFort besoin',
        fontsize=8, color='orange', va='top')
ax.text(xmin, my * 0.5, 'Faible marcha.\nFaible besoin',
        fontsize=8, color='steelblue', va='top')
ax.text(mx * 1.01, my * 0.5, 'Bonne marcha.\nFaible besoin',
        fontsize=8, color='green', va='top')

ax.set_xlabel('Walk Index')
ax.set_ylabel('Score de précarité (precarite_score_24)')
ax.set_title('Marchabilité vs Précarité par sous-secteur GIREC')
ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left', title='Commune')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# ← background zones_girec en gris clair
zones_girec.plot(ax=ax, color="white", edgecolor=(0.5, 0.5, 0.5, 0.5), linewidth=0.5)

# ← plot principal par dessus
zones_girec.dropna(subset=["precarite_score_24"]).plot(
    column="precarite_score_24",
    cmap="YlOrRd",
    linewidth=0.5,
    edgecolor="grey",
    legend=True,
    ax=ax,
    legend_kwds={"label": "Score de précarité (brut)", "shrink": 0.6}
)

#canton_GE.boundary.plot(ax=ax, color="black", linewidth=1.5)
ax.set_title("Score de précarité par zone GIREC (non normalisé)", fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.show()

### 2. Score de priorité — faible marchabilité × fort besoin social

In [ ]:
scaler = MinMaxScaler()
df_prio = zones_girec[['walk_index', 'walk_index_norm',  'precarite_score_24', 'NOM', 'NO_COM_FED', 'COMMUNE', 'geometry']].dropna().copy()

# precarite_score_24 normalisé 0-1 (plus c'est élevé = plus précaire = plus besoin d'action)
df_prio['precarity_norm'] = scaler.fit_transform(df_prio[['precarite_score_24']])

# Priority score : fort quand marchabilité FAIBLE et précarité ÉLEVÉE
df_prio['priority_score'] = (1 - df_prio['walk_index_norm']) * df_prio['precarity_norm']

# Top 20 zones prioritaires
top20 = df_prio.nlargest(20, 'priority_score')[['NOM', 'COMMUNE', 'walk_index', 'walk_index_norm', 'precarite_score_24', 'precarity_norm', 'priority_score']]
print(f'{len(df_prio)} sous-secteurs avec données complètes / {len(zones_girec)} total')
print()
print('Top 20 zones prioritaires :')
print(top20.to_string(index=False))

In [ ]:
df_prio.isna().sum()

### 3. Carte — score de priorité

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

gdf_prio = gpd.GeoDataFrame(df_prio, geometry='geometry', crs=zones_girec.crs)

fig, ax = plt.subplots(figsize=(12, 10))

# Fond : délimitation de tous les sous-secteurs (y compris sans données)
zones_girec.to_crs(gdf_prio.crs).plot(
    ax=ax, color='whitesmoke', edgecolor='lightgrey', linewidth=0.5
)

# Dessus : score de priorité
gdf_prio.plot(
    column='priority_score',
    cmap='RdYlGn_r',
    legend=True,
    legend_kwds={'label': 'Score de priorité', 'shrink': 0.6},
    ax=ax
)

ax.set_title('Zones prioritaires — faible marchabilité × fort besoin social', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
#replace les colonnes créées au bon endroit dans zones_girec

# walk_index_norm n'est PAS modifié dans df_prio — ne pas le réassigner
# (df_prio a été créé avec dropna, ce qui introduirait 124 NaN par alignement d'index)
zones_girec['precarity_norm'] = df_prio['precarity_norm']
zones_girec['priority_score'] = df_prio['priority_score']

cols = zones_girec.columns.tolist()
for col, after in [
    ('walk_index_norm',      'walk_index'),
    ('precarity_norm', 'precarite_score_24'),
    ('priority_score', 'precarity_norm')
]:
    cols.insert(cols.index(after) + 1, cols.pop(cols.index(col)))
zones_girec = zones_girec[cols]

In [ ]:
zones_girec.columns

### 4. Clustering K-Means — profils de sous-secteurs

4 clusters : bonne marcha./aisé · bonne marcha./précaire · mauvaise marcha./aisé · mauvaise marcha./précaire

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd

cluster_vars = ['walk_index_norm', 'precarite_score_24']
df_clust = zones_girec[['NOM', 'NO_COM_FED', 'geometry'] + cluster_vars].dropna().copy()

scaler_k = StandardScaler()
X_clust = scaler_k.fit_transform(df_clust[cluster_vars])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_clust['cluster'] = kmeans.fit_predict(X_clust)

# Profil moyen de chaque cluster
profile = df_clust.groupby('cluster')[cluster_vars].mean().round(3)
print("Profil moyen par cluster :")
print(profile)
print()
print("Nombre de sous-secteurs par cluster :")
print(df_clust['cluster'].value_counts().sort_index())

In [ ]:
# Nommer les clusters selon leur profil
centers = df_clust.groupby('cluster')[['walk_index_norm', 'precarite_score_24']].mean()
labels = {}
for i, row in centers.iterrows():
    walk_high = row['walk_index_norm'] > centers['walk_index_norm'].median()
    prec_high = row['precarite_score_24'] > centers['precarite_score_24'].median()
    if walk_high and prec_high:
        labels[i] = 'Bonne marcha. · Précaire'
    elif walk_high and not prec_high:
        labels[i] = 'Bonne marcha. · Aisé'
    elif not walk_high and prec_high:
        labels[i] = 'Faible marcha. · Précaire ⚠︎'
    else:
        labels[i] = 'Faible marcha. · Aisé'

df_clust['cluster_label'] = df_clust['cluster'].map(labels)

# ─── Afficher labels + valeurs moyennes des clusters ──────────────────────────
print("── Clusters avec valeurs moyennes ───────────────────")
for i, label in labels.items():
    row = centers.loc[i]
    print(f"  {label:<35} walk_index_norm={row['walk_index_norm']:.3f}  precarite={row['precarite_score_24']:.3f}")

print()
print("── Nombre de zones par cluster ──────────────────────")
print(df_clust['cluster_label'].value_counts())

In [ ]:
df_clust

In [ ]:
df_clust.isna().sum()

In [ ]:
# ─── Liste des zones "Faible marcha. · Précaire" avec stats du cluster ────────
mask = df_clust["cluster_label"] == "Faible marcha. · Précaire ⚠︎"
subset = df_clust[mask]

# Stats du cluster
print(f"── Cluster : Faible marcha. · Précaire ⚠︎ ({mask.sum()} zones) ────────")
print(f"   walk_index moyen      : {subset['walk_index_norm'].mean():.3f}")
print(f"   precarite_score moyen : {subset['precarite_score_24'].mean():.3f}")
print(f"   walk_index médian     : {subset['walk_index_norm'].median():.3f}")
print(f"   precarite_score médian: {subset['precarite_score_24'].median():.3f}")
print()

# Liste des zones
print(f"── Liste des zones ───────────────────────────────────────────────────")
print(subset[["NOM", "walk_index_norm", "precarite_score_24"]]
      .sort_values("precarite_score_24", ascending=False)
      .to_string(index=False))

### 5. Carte des clusters

In [ ]:
import matplotlib.patches as mpatches

gdf_clust = gpd.GeoDataFrame(df_clust, geometry='geometry', crs=zones_girec.crs)

cluster_colors = {
    'Bonne marcha. · Aisé':        '#a2c29fff',
    'Bonne marcha. · Précaire':    '#e7c481ff',
    'Faible marcha. · Aisé':       '#e7c481ff',
    'Faible marcha. · Précaire ⚠︎': '#e3bbb1ff'
}

gdf_clust['color'] = gdf_clust['cluster_label'].map(cluster_colors).fillna('lightgrey')

fig, ax = plt.subplots(figsize=(13, 11))

# Fond : tous les sous-secteurs (zones sans données = whitesmoke)
zones_girec.to_crs(gdf_clust.crs).plot(
    ax=ax, color='whitesmoke', edgecolor='lightgrey', linewidth=0.5
)

# Dessus : clusters colorés
gdf_clust.plot(color=gdf_clust['color'], ax=ax, edgecolor='white', linewidth=0.3)

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in cluster_colors.items()]
legend_patches.append(mpatches.Patch(color='whitesmoke', edgecolor='lightgrey', label='Sans données'))
ax.set_title('Profils des sous-secteurs GIREC — marchabilité × précarité', fontsize=13)
ax.set_axis_off()

ax.legend(
    handles=legend_patches,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.02),  # ← en dessous du plot
    ncol=2,
    fontsize=9,
    title="Profil",
    frameon=True
)

plt.tight_layout()
plt.show()

In [ ]:
zones_girec['cluster'] = df_clust['cluster']
zones_girec['cluster_label'] = df_clust['cluster_label']

cols = zones_girec.columns.tolist()
for col, after in [
    ('cluster',      'priority_score'),
    ('cluster_label', 'cluster')
]:
    cols.insert(cols.index(after) + 1, cols.pop(cols.index(col)))
zones_girec = zones_girec[cols]


In [ ]:
zones_girec.columns

### 6. Scatter walk_index × revenus médian coloré par cluster

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for label, grp in gdf_clust.groupby('cluster_label'):
    ax.scatter(grp['walk_index_norm'], grp['precarite_score_24'],
               color=cluster_colors[label], label=label, alpha=0.75, s=45)

ax.axvline(gdf_clust['walk_index_norm'].median(), color='gray', linestyle='--', linewidth=0.8)
ax.axhline(gdf_clust['precarite_score_24'].median(), color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('walk_index_norm')
ax.set_ylabel('Score de précarité (precarite_score_24)')
ax.set_title('Clusters GIREC — walk_index × précarité')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Centres des clusters dans l'espace original (non standardisé) ────────────
centers_original = df_clust.groupby('cluster_label')[cluster_vars].mean().round(3)
print("── Centres des clusters (valeurs originales) ────────")
print(centers_original)
print()

# ─── Valeurs min/max de chaque cluster ────────────────────────────────────────
print("── Plages de valeurs par cluster ────────────────────")
for label in df_clust['cluster_label'].unique():
    subset = df_clust[df_clust['cluster_label'] == label]
    print(f"\n{label} ({len(subset)} zones) :")
    print(f"  walk_index       : {subset['walk_index_norm'].min():.3f} → {subset['walk_index_norm'].max():.3f}  (mean: {subset['walk_index_norm'].mean():.3f})")
    print(f"  precarite_score  : {subset['precarite_score_24'].min():.3f} → {subset['precarite_score_24'].max():.3f}  (mean: {subset['precarite_score_24'].mean():.3f})")

### MARCHABILITE X NOMBRE VOITURES POSSEDEE

In [ ]:
df_voit = zones_girec[['walk_index_norm', 'freq_voitures_24', 'NOM', 'COMMUNE', 'geometry']].dropna().copy()

scaler = MinMaxScaler()
df_voit['voiture_norm'] = scaler.fit_transform(df_voit[['freq_voitures_24']])

# Score : fort quand FAIBLE marchabilité ET FAIBLE fréquence voiture
df_voit['priority_voiture'] = (1 - df_voit['walk_index_norm']) * (1 - df_voit['voiture_norm'])

print(f'{len(df_voit)} sous-secteurs avec données complètes')
print()
top20 = df_voit.nlargest(20, 'priority_voiture')[['NOM', 'COMMUNE', 'walk_index_norm', 'freq_voitures_24', 'voiture_norm', 'priority_voiture']]
print('Top 20 zones prioritaires (faible marcha. + faible taux voiture) :')
print(top20.to_string(index=False))

In [ ]:
# ─── Vérifier les NaN par colonne ─────────────────────────────────────────────
cols_check = ['walk_index_norm', 'freq_voitures_24', 'NOM', 'COMMUNE']

print(f"Total sous-secteurs : {len(zones_girec)}")
print()
for col in cols_check:
    n_nan = zones_girec[col].isna().sum()
    print(f"  {col:<25} NaN : {n_nan} ({n_nan/len(zones_girec)*100:.1f}%)")

print(f"\nAprès dropna : {zones_girec[cols_check].dropna().shape[0]} sous-secteurs")

In [ ]:
# ─── Calcul des médianes ────────────────────────────────────
mx = df_voit['walk_index_norm'].median()
my = df_voit['freq_voitures_24'].median()

# ─── Palette couleurs quadrants ───────────────────────────────────────────────
quadrant_colors = {
    'faible_marcha_peu_voitures'   : '#e3bbb1ff',  # ← PRIORITAIRE
    'bonne_marcha_peu_voitures'    : '#a2c29fff',
    'faible_marcha_bcp_voitures'   : '#e7c481ff',
    'bonne_marcha_bcp_voitures'    : '#e7c481ff',
}

# ─── Fonction quadrant adaptée ────────────────────────────────────────────────
def quadrant_color(row):
    if row['walk_index_norm'] < mx and row['freq_voitures_24'] < my:
        return quadrant_colors['faible_marcha_peu_voitures']
    elif row['walk_index_norm'] >= mx and row['freq_voitures_24'] < my:
        return quadrant_colors['bonne_marcha_peu_voitures']
    elif row['walk_index_norm'] < mx and row['freq_voitures_24'] >= my:
        return quadrant_colors['faible_marcha_bcp_voitures']
    else:
        return quadrant_colors['bonne_marcha_bcp_voitures']

df_voit['quad_color'] = df_voit.apply(quadrant_color, axis=1)

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(df_voit['walk_index_norm'], df_voit['freq_voitures_24'],
           c=df_voit['quad_color'], alpha=0.7, s=40)

ax.axvline(mx, color='gray', linestyle='--', linewidth=0.8)
ax.axhline(my, color='gray', linestyle='--', linewidth=0.8)

ax.text(mx, ax.get_ylim()[0], f'médiane : {mx:.3f}',
        color='gray', fontsize=8, ha='center', va='bottom')
ax.text(ax.get_xlim()[0], my, f'médiane {my:.3f}',
        color='gray', fontsize=8, ha='left', va='center')

xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

ax.set_xlabel('walk_index_norm')
ax.set_ylabel('Fréquence voitures (pour 1000 hab.)')
ax.set_title('Marchabilité vs Fréquence voitures — zones GIREC')
plt.tight_layout()
plt.show()

In [ ]:
# Carte choroplèthe — score priorité voiture
gdf_voit = gpd.GeoDataFrame(df_voit, geometry='geometry', crs=zones_girec.crs)

fig, ax = plt.subplots(figsize=(12, 10))

zones_girec.to_crs(gdf_voit.crs).plot(
    ax=ax, color='whitesmoke', edgecolor='lightgrey', linewidth=0.5
)
gdf_voit.plot(
    column='priority_voiture',
    cmap='RdYlGn_r',
    legend=True,
    legend_kwds={'label': 'Score priorité (faible marcha. × faible taux voiture)', 'shrink': 0.6},
    ax=ax
)
ax.set_title('Zones prioritaires — faible marchabilité × faible taux de motorisation', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
zones_girec['voiture_norm'] = df_voit['voiture_norm']
zones_girec['priority_voiture'] = df_voit['priority_voiture']

cols = zones_girec.columns.tolist()
for col, after in [
    ('voiture_norm',      'freq_voitures_24'),
    ('priority_voiture', 'voiture_norm')
]:
    cols.insert(cols.index(after) + 1, cols.pop(cols.index(col)))
zones_girec = zones_girec[cols]

In [ ]:
zones_girec.columns

### DESSERT TP - SOUS-SECTEUR GIREC

In [ ]:
# Qualite de desserte TP — % de recouvrement par classe pour chaque sous-secteur GIREC
dessert_tp = gpd.read_file(
    f'{input_file_path}/attributs/GE/dessert_TP/gueteklassen_oev_2023_2056.gpkg/OeV_Gueteklassen_ARE.gpkg',
    layer='OeV_Gueteklassen_ARE'
)[['KLASSE', 'KLASSE_FR', 'geometry']]

# Intersection between GIREC zones and dessert TP polygons
zones_proj = zones_girec[['CODE', 'geometry']].copy().to_crs("EPSG:2056")
zones_proj['area_zone'] = zones_proj.geometry.area

intersect = gpd.overlay(zones_proj, dessert_tp.to_crs("EPSG:2056"), how='intersection')
intersect['area_inter'] = intersect.geometry.area

# Sum intersection area per zone x class
dessert_by_zone = (
    intersect
    .groupby(['CODE', 'KLASSE'])['area_inter']
    .sum()
    .reset_index()
)

# Zone total areas
zone_areas = zones_proj[['CODE', 'area_zone']].drop_duplicates()

# Compute % coverage per class per zone
dessert_by_zone = dessert_by_zone.merge(zone_areas, on='CODE')
dessert_by_zone['pct'] = (dessert_by_zone['area_inter'] / dessert_by_zone['area_zone'] * 100).round(1)

# Pivot: one column per KLASSE
dessert_pivot = (
    dessert_by_zone.pivot(index='CODE', columns='KLASSE', values='pct')
    .fillna(0)
    .reset_index()
)
# Rename columns: desserte_pct_A, desserte_pct_B, etc.
dessert_pivot.columns = (
    ['CODE'] + [f'desserte_pct_{col}' for col in dessert_pivot.columns[1:]]
)

print(f"Zones GIREC: {len(dessert_pivot)} / {len(zones_girec)}")
print(f"Classes trouvees: {[c for c in dessert_pivot.columns if c != 'CODE']}")
print()

# Check: sum of all class % per zone (may be < 100 if zone partially outside desserte coverage)
pct_cols = [c for c in dessert_pivot.columns if c.startswith('desserte_pct_')]
dessert_pivot['desserte_pct_total_couvert'] = dessert_pivot[pct_cols].sum(axis=1).round(1)

print("Distribution du % de recouvrement total par zone:")
print(dessert_pivot['desserte_pct_total_couvert'].describe().round(1))
print()
print("Apercu:")
display(dessert_pivot.head())

# Merge into zones_girec
zones_girec = zones_girec.merge(dessert_pivot, on='CODE', how='left')

In [ ]:
# ─── Pondération ARE / SN 640 281 ─────────────────────────────────────────────
# Source : méthodologie ARE (Office fédéral du développement territorial)
# Poids linéaires : A=1.0, B=0.75, C=0.5, D=0.25
weights = {
    'A': 1.00,
    'B': 0.75,
    'C': 0.50,
    'D': 0.25,
}

# ─── Calcul du score de desserte ──────────────────────────────────────────────
# Les % sont entre 0 et 100 → on divise par 100 pour normaliser entre 0 et 1
zones_girec["desserte_score"] = (
    zones_girec["desserte_pct_A"] * weights['A'] +
    zones_girec["desserte_pct_B"] * weights['B'] +
    zones_girec["desserte_pct_C"] * weights['C'] +
    zones_girec["desserte_pct_D"] * weights['D']
) / 100

print("── Transport score distribution ──────────────────────")
print(zones_girec["desserte_score"].describe().round(3))
print(f"\nMin : {zones_girec['desserte_score'].min():.3f}")
print(f"Max : {zones_girec['desserte_score'].max():.3f}")

In [ ]:
zones_girec

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# ─── Background zones sans données ───────────────────────────────────────────
zones_girec.plot(ax=ax, color="white", edgecolor=(0.5, 0.5, 0.5, 0.3), linewidth=0.5)

# ─── Plot transport score ──────────────────────────────────────────────────────
zones_girec.dropna(subset=["desserte_score"]).plot(
    column="desserte_score",
    cmap="RdYlGn",
    linewidth=0.5,
    edgecolor=(0.5, 0.5, 0.5, 0.3),
    legend=True,
    ax=ax,
    vmin=0, vmax=1,
    legend_kwds={"label": "Score de desserte TP (0-1)", "shrink": 0.6}
)

ax.set_title(
    "Score de desserte en transports publics par zone GIREC\n"
    "(pondération A=1.0, B=0.75, C=0.5, D=0.25)",
    fontweight="bold", fontsize=11
)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Distribution du score de desserte TP — Zones GIREC", fontweight="bold")

data = zones_girec["desserte_score"].dropna()

# ─── Histogramme ──────────────────────────────────────────────────────────────
axes[0].hist(data, bins=30, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].axvline(data.median(), color="red",    linestyle="--", label=f"Médiane: {data.median():.3f}")
axes[0].axvline(data.mean(),   color="orange", linestyle="--", label=f"Moyenne: {data.mean():.3f}")
axes[0].set_title("Histogramme", fontweight="bold")
axes[0].set_xlabel("Score de desserte TP (0-1)")
axes[0].set_ylabel("Nombre de zones")
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.3)

# ─── Boxplot ──────────────────────────────────────────────────────────────────
bp = axes[1].boxplot(data.values, vert=True, patch_artist=True, widths=0.5,
                     medianprops=dict(color="red", linewidth=2),
                     flierprops=dict(marker="o", markersize=3, alpha=0.4))
bp["boxes"][0].set_facecolor("#4C72B0")
bp["boxes"][0].set_alpha(0.75)
axes[1].axhline(data.mean(), color="orange", linestyle="--", linewidth=1.2)
axes[1].text(1.32, data.median(), f"Médiane: {data.median():.3f}", va="center", color="red",    fontsize=9)
axes[1].text(1.32, data.mean(),   f"Moyenne: {data.mean():.3f}",   va="center", color="orange", fontsize=9)
axes[1].set_title("Boxplot", fontweight="bold")
axes[1].set_ylabel("Score de desserte TP (0-1)")
axes[1].set_xticks([])
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("── Stats desserte_score ──────────────────────────────")
print(data.describe().round(3))

In [ ]:
# ─── Calcul de la surface de chaque sous-secteur ──────────────────────────────
zones_girec["area"] = zones_girec.geometry.area  # en m²

data_weighted = zones_girec[["desserte_score", "area"]].dropna()

total_area = data_weighted["area"].sum()
data_weighted["pct_area"] = data_weighted["area"] / total_area * 100

print(f"Surface totale canton : {total_area/1e6:.1f} km²")
print(f"Zones avec données    : {len(data_weighted)}")

# ─── Distribution pondérée par surface ────────────────────────────────────────
# % de surface du canton par tranche de score
bins = np.arange(0, 1.05, 0.1)
labels_bins = [f"{b:.1f}-{b+0.1:.1f}" for b in bins[:-1]]

data_weighted["score_bin"] = pd.cut(data_weighted["desserte_score"], bins=bins, labels=labels_bins)

area_by_bin = (data_weighted
               .groupby("score_bin", observed=True)["area"]
               .sum()
               .reset_index())
area_by_bin["pct_area"] = area_by_bin["area"] / total_area * 100

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Distribution du score de desserte TP\npondérée par surface des sous-secteurs",
             fontweight="bold")

# ── Histogramme pondéré par surface ───────────────────────────────────────────
axes[0].bar(range(len(area_by_bin)), area_by_bin["pct_area"],
            color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].set_xticks(range(len(area_by_bin)))
axes[0].set_xticklabels(area_by_bin["score_bin"], rotation=45, ha="right", fontsize=8)
axes[0].set_title("% de surface du canton par tranche de score", fontweight="bold")
axes[0].set_xlabel("Score de desserte TP")
axes[0].set_ylabel("% de surface du canton")
axes[0].grid(axis="y", alpha=0.3)

# Médiane pondérée
median_w = np.average(data_weighted["desserte_score"],
                      weights=data_weighted["area"])
axes[0].axvline(
    next(i for i, b in enumerate(labels_bins) if float(b.split('-')[0]) >= median_w),
    color="orange", linestyle="--", linewidth=1.5,
    label=f"Moyenne pondérée : {median_w:.3f}"
)
axes[0].legend(fontsize=9)

# ── Comparaison non pondéré vs pondéré ────────────────────────────────────────
mean_unweighted = data_weighted["desserte_score"].mean()
mean_weighted   = np.average(data_weighted["desserte_score"], weights=data_weighted["area"])

categories = ["Moyenne\n(par sous-secteur)", "Moyenne pondérée\n(par surface)"]
values     = [mean_unweighted, mean_weighted]
colors     = ["#DD8452", "#4C72B0"]

bars = axes[1].bar(categories, values, color=colors, edgecolor="white", alpha=0.85, width=0.4)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

axes[1].set_title("Comparaison moyenne simple vs pondérée par surface",
                  fontweight="bold")
axes[1].set_ylabel("Score de desserte TP (0-1)")
axes[1].set_ylim(0, 1)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# ─── Print résumé ─────────────────────────────────────────────────────────────
print(f"\nMoyenne simple (par sous-secteur) : {mean_unweighted:.3f}")
print(f"Moyenne pondérée (par surface)    : {mean_weighted:.3f}")
print(f"\n% de surface par tranche :")
print(area_by_bin[["score_bin", "pct_area"]].to_string(index=False))

In [ ]:
zones_girec["priority_dessert"] = (
    (1 - zones_girec["walk_index_norm"]) *
    (1 - zones_girec["voiture_norm"]) *
    (1 - zones_girec["desserte_score"])
)

print("\n── priority_dessert ──────────────────────────────────")
print(zones_girec["priority_dessert"].describe().round(3))
print(f"\nTop 5 zones prioritaires :")
print(zones_girec.nlargest(5, "priority_dessert")[["NOM", "COMMUNE", "walk_index_norm", "voiture_norm", "desserte_score", "priority_dessert"]].to_string(index=False))

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))

zones_girec.plot(ax=ax, color="white", edgecolor=(0.5, 0.5, 0.5, 0.3), linewidth=0.5)

zones_girec.dropna(subset=["priority_dessert"]).plot(
    column="priority_dessert",
    cmap="RdYlGn_r",  # ← rouge = prioritaire, vert = pas urgent
    linewidth=0.5,
    edgecolor=(0.5, 0.5, 0.5, 0.3),
    legend=True,
    ax=ax,
    legend_kwds={
        "label" : "Score de priorité\n(marchabilité × motorisation × desserte TP)",
        "shrink": 0.6
    }
)

ax.set_title(
    "Zones prioritaires — Faible marchabilité × Faible motorisation × Faible desserte TP\n"
    "score = (1-walk_index) × (1-voiture_norm) × (1-transport_score)",
    fontweight="bold", fontsize=11
)
ax.set_axis_off()
plt.tight_layout()
plt.show()

### DESSERTE TP - AGGLO_CARREAUX

In [ ]:
# ─── Intersection entre agglo_carreau et dessert TP ───────────────────────────
carreau_proj = agglo_carreau[['GRID_ID', 'geometry']].copy().to_crs("EPSG:2056")
carreau_proj['area_zone'] = carreau_proj.geometry.area

intersect_carreau = gpd.overlay(carreau_proj, dessert_tp.to_crs("EPSG:2056"), how='intersection')
intersect_carreau['area_inter'] = intersect_carreau.geometry.area

# ─── Sum intersection area per carreau x class ────────────────────────────────
dessert_by_carreau = (
    intersect_carreau
    .groupby(['GRID_ID', 'KLASSE'])['area_inter']
    .sum()
    .reset_index()
)

# ─── Zone total areas ──────────────────────────────────────────────────────────
carreau_areas = carreau_proj[['GRID_ID', 'area_zone']].drop_duplicates()

# ─── Compute % coverage per class per carreau ─────────────────────────────────
dessert_by_carreau = dessert_by_carreau.merge(carreau_areas, on='GRID_ID')
dessert_by_carreau['pct'] = (
    dessert_by_carreau['area_inter'] / dessert_by_carreau['area_zone'] * 100
).round(1)

# ─── Pivot ────────────────────────────────────────────────────────────────────
dessert_pivot_carreau = (
    dessert_by_carreau.pivot(index='GRID_ID', columns='KLASSE', values='pct')
    .fillna(0)
    .reset_index()
)

dessert_pivot_carreau.columns = (
    ['GRID_ID'] + [f'desserte_pct_{col}' for col in dessert_pivot_carreau.columns[1:]]
)

# ─── Check ────────────────────────────────────────────────────────────────────
pct_cols = [c for c in dessert_pivot_carreau.columns if c.startswith('desserte_pct_')]
dessert_pivot_carreau['desserte_pct_total_couvert'] = dessert_pivot_carreau[pct_cols].sum(axis=1).round(1)

print(f"Carreaux avec desserte : {len(dessert_pivot_carreau)} / {len(agglo_carreau)}")
print(f"Classes trouvées       : {pct_cols}")
print()
print("Distribution du % de recouvrement total par carreau:")
print(dessert_pivot_carreau['desserte_pct_total_couvert'].describe().round(1))

# ─── Merge into agglo_carreau ─────────────────────────────────────────────────
agglo_carreau = agglo_carreau.merge(dessert_pivot_carreau, on='GRID_ID', how='left')

# ─── Transport score pour chaque carreau ──────────────────────────────────────
weights = {'A': 1.00, 'B': 0.75, 'C': 0.50, 'D': 0.25}

agglo_carreau["desserte_score"] = (
    agglo_carreau.get("desserte_pct_A", 0) * weights['A'] +
    agglo_carreau.get("desserte_pct_B", 0) * weights['B'] +
    agglo_carreau.get("desserte_pct_C", 0) * weights['C'] +
    agglo_carreau.get("desserte_pct_D", 0) * weights['D']
) / 100

In [ ]:
agglo_carreau

In [ ]:
# ─── Score marchabilité × desserte ───────────────────────────────────────────
agglo_carreau["score_marcha_desserte"] = (
    (1 - agglo_carreau["walk_index_norm"]) * agglo_carreau["desserte_score"]
)

fig, ax = plt.subplots(figsize=(12, 12))

agglo_carreau.plot(ax=ax, color="white", edgecolor=(0.5, 0.5, 0.5, 0.1), linewidth=0.3)

agglo_carreau.dropna(subset=["score_marcha_desserte"]).plot(
    column="score_marcha_desserte",
    cmap="RdYlGn_r",
    linewidth=0.3,
    edgecolor=(0.5, 0.5, 0.5, 0.1),
    legend=True,
    ax=ax,
    legend_kwds={
        "label" : "Score marchabilité × desserte TP",
        "shrink": 0.6
    }
)

canton_GE.boundary.plot(ax=ax, color="black", linewidth=1.5)
lac_clipped = gpd.clip(lac.to_crs(canton_GE.crs), canton_GE)
lac_clipped.plot(ax=ax, color="lightblue", edgecolor=(0.5, 0.5, 0.5, 0.3), linewidth=0.5)



ax.set_title(
    "Marchabilité × Desserte TP — Agglo Carreau \n" \
    "Forte desserte TP mais faible marchabilité -> zone desserte sous exploitée",
    fontweight="bold", fontsize=11
)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
agglo_carreau

### SECURITE/INSECURITE

# EXPORTS

In [ ]:
zones_girec.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_girec.gpkg"), driver="GPKG")
zones_girec.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_girec.parquet')
zones_girec.to_csv(f'{output_step3_path}/step3_aggregated_index_girec.csv')
print("zones_girec exported as .gpkg, .parquet, .csv")

agglo_carreau.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_carreau200.gpkg"), driver="GPKG")
agglo_carreau.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_carreau200.parquet')
print("agglo_carreau exported as .gpkg, .parquet")

zones_communes.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes.gpkg"), driver="GPKG")
zones_communes.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes.parquet')
print("zones_communes exported as .gpkg, .parquet")

zones_communes_GE_fusionnee.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes_GE_fusionnee.gpkg"), driver="GPKG")
zones_communes_GE_fusionnee.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes_GE_fusionnee.parquet')
print("zones_commune_GE_fusionne exported as .gpkg, .parquet")